# 第 3 章互动实验：结构元素如何改变形状

实验使用合成二值图和 OpenCV 基线，只用于观察，不提供 `3-1_手写二值形态学.py` 的循环实现。重点关注前景像素数、孔洞、间隙和错误粘连。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, IntSlider

binary = np.zeros((120, 180), dtype=np.uint8)
binary[25:90, 20:65] = 255
binary[43:70, 35:50] = 0                 # 孔洞
binary[35:80, 82:120] = 255
binary[35:80, 125:162] = 255             # 5 像素间隙
rng = np.random.default_rng(2026)
ys = rng.integers(0, binary.shape[0], 35)
xs = rng.integers(0, binary.shape[1], 35)
binary[ys, xs] = 255

In [ ]:
OPERATIONS = {
    '腐蚀': cv2.MORPH_ERODE, '膨胀': cv2.MORPH_DILATE,
    '开运算': cv2.MORPH_OPEN, '闭运算': cv2.MORPH_CLOSE,
}

def show_morphology(operation='闭运算', width=7, height=3):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (width, height))
    code = OPERATIONS[operation]
    if code == cv2.MORPH_ERODE:
        result = cv2.erode(binary, kernel)
    elif code == cv2.MORPH_DILATE:
        result = cv2.dilate(binary, kernel)
    else:
        result = cv2.morphologyEx(binary, code, kernel)
    n0, _, _, _ = cv2.connectedComponentsWithStats(binary, 8)
    n1, _, _, _ = cv2.connectedComponentsWithStats(result, 8)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(binary, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'原图：前景 {np.count_nonzero(binary)}，区域 {n0-1}')
    axes[1].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[1].set_title(f'{operation} {width}×{height}：前景 {np.count_nonzero(result)}，区域 {n1-1}')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_morphology,
    operation=Dropdown(options=list(OPERATIONS), value='闭运算'),
    width=IntSlider(min=1, max=21, step=2, value=7),
    height=IntSlider(min=1, max=15, step=2, value=3),
);

## 破坏实验与记录

- 用窄高核和宽扁核分别做闭运算，比较方向先验。
- 增大开运算核，找到主体开始被误删的临界尺寸。
- 找到使右侧两个矩形错误粘连的最小核宽。

把前景像素数、连通域数和失败解释写入 `实验记录.md`。